In [2]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE


data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
])


qualitative_features = input_data.select_dtypes(include=["object", "string", "category"]).columns.tolist() 
#print(qualitative_features) => ['commit_cadence', 'primary_language', 'framework', 'has_custom_art', 'hosting_type', 'is_mobile_responsive', 'project_category', 'is_multiplayer', 'mentions_ai_api', 'is_clone_or_tutorial', 'author_prior_projects', 'author_account_age_years']

for col in qualitative_features: #fill in missing stuff just in case
    input_data[col] = input_data[col].fillna("Unknown").astype(str)

input_training, input_validation, output_training, output_validation = train_test_split(input_data, target_thing, test_size=0.2, random_state=42)

magicModel = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.01,
    depth=6,
    loss_function="RMSE",
    verbose=100,
    early_stopping_rounds=100
)

magicModel.fit(
    input_training,
    output_training,
    cat_features=qualitative_features,
    eval_set=(input_validation, output_validation)
)

predictions=np.expm1(magicModel.predict(input_validation))
predictions = np.maximum(predictions, 0) #no negatives

true_hours = np.expm1(output_validation)

score=RMSE(true_hours,predictions)

print("the score:", score)

#diagnostics

how_important = pd.DataFrame({
    "feature": input_data.columns,
    "importance": magicModel.feature_importances_
})

print("correlations")
print(how_important.head(15).to_string(index=False))

0:	learn: 0.9725603	test: 0.9021335	best: 0.9021335 (0)	total: 78.6ms	remaining: 1m 18s
100:	learn: 0.7644205	test: 0.7292849	best: 0.7292849 (100)	total: 620ms	remaining: 5.52s
200:	learn: 0.6961994	test: 0.6889588	best: 0.6889588 (200)	total: 1.21s	remaining: 4.83s
300:	learn: 0.6648293	test: 0.6778676	best: 0.6778676 (300)	total: 1.81s	remaining: 4.2s
400:	learn: 0.6455286	test: 0.6726811	best: 0.6726811 (400)	total: 2.31s	remaining: 3.46s
500:	learn: 0.6311840	test: 0.6695362	best: 0.6695362 (500)	total: 2.79s	remaining: 2.78s
600:	learn: 0.6197327	test: 0.6673628	best: 0.6673031 (598)	total: 3.35s	remaining: 2.23s
700:	learn: 0.6106168	test: 0.6656459	best: 0.6656459 (700)	total: 3.86s	remaining: 1.65s
800:	learn: 0.6012563	test: 0.6643841	best: 0.6643634 (797)	total: 4.33s	remaining: 1.07s
900:	learn: 0.5920519	test: 0.6631965	best: 0.6631965 (900)	total: 4.78s	remaining: 526ms
999:	learn: 0.5839420	test: 0.6621718	best: 0.6621718 (999)	total: 5.28s	remaining: 0us

bestTest = 0.6

In [ ]:
import pandas as pd
import numpy as np

data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

print("Projects per categoy")
print(data["project_category"].value_counts())

print("average horus per category")
print(data.groupby("project_category")["hours_spent"]
      .agg(["count", "mean", "median","std", "min", "max"])
      .sort_values("count", ascending=False))

Projects per categoy
project_category
game       1687
app         961
website     939
tool        697
bot         128
other        83
Name: count, dtype: int64
average horus per category
                  count       mean  median        std  min    max
project_category                                                 
game               1687  11.486011     6.0  18.495224  0.1  140.7
app                 961  14.663684     8.0  19.814738  0.2  140.7
website             939   7.617359     3.1  13.445596  0.1  140.7
tool                697  16.230560     8.8  24.041063  0.1  140.7
bot                 128  12.771094     7.3  19.631339  0.2  140.7
other                83  12.039759     6.0  15.518376  0.7   85.0
